# 12 (DS) — Feature Engineering Pipeline

**Data Scientist perspective.** From raw orders to a model-ready feature table: cleaning, date-part extraction, conditional bucketing, window features (deltas, rolling windows), an ML-transformer chain, and materialization via `saveAsTable`. Each stage is lazy — `lineage` records the full recipe.

In [ ]:
import os
from dotenv import load_dotenv
from irispark import IrisParkSession

load_dotenv()

# Connection via environment variables (matches examples/basic_usage.py).
# Set IRIS_HOST / IRIS_PORT / IRIS_NAMESPACE / IRIS_USERNAME / IRIS_PASSWORD.
try:
    session = IrisParkSession.builder() \
        .host(os.environ.get("IRIS_HOST", "localhost")) \
        .port(int(os.environ.get("IRIS_PORT", 1972))) \
        .namespace(os.environ.get("IRIS_NAMESPACE", "USER")) \
        .username(os.environ.get("IRIS_USERNAME", "_SYSTEM")) \
        .password(os.environ.get("IRIS_PASSWORD", "SYS")) \
        .getOrCreate()
    print("Connected to IRIS:", session)
except Exception as e:
    print("SKIP: IRIS not reachable -", e)
    session = None

In [ ]:
if session is None:
    raise SystemExit("IRIS not reachable; skipping this notebook.")

## 1. Raw events

Orders with dates, categories, amounts — plus duplicates and NULLs to clean.

In [ ]:
import random
import pandas as pd

random.seed(7)
CATS = ["eletronico", "moveis", "vestuario", "alimentos"]
rows = []
for i in range(1, 301):
    d = f"2025-{random.randint(1, 6):02d}-{random.randint(1, 28):02d}"
    rows.append({
        "pedido_id": i,
        "cliente_id": random.randint(1, 60),
        "data_pedido": d,
        "categoria": random.choice(CATS),
        "valor": round(random.uniform(20, 900), 2) if i % 17 else None,
    })
df = session.createDataFrame(pd.DataFrame(rows))
# plant two exact duplicates
df = df.union(df.filter("pedido_id IN (5, 10)"))
print("raw rows:", df.count())
df.show(5)

## 2. Cleaning pass

Deduplicate on the business key, drop rows without amount (they cannot be trained on).

In [ ]:
clean = df.dropDuplicates(["pedido_id"]).filter("valor IS NOT NULL")
print("clean rows:", clean.count())

## 3. Date-part features

Calendar decomposition via IRIS date functions.

In [ ]:
from irispark.functions import col, month, quarter, dayofmonth, dayofweek, monthname

feats = clean.withColumn("mes", month("data_pedido")) \
             .withColumn("trimestre", quarter("data_pedido")) \
             .withColumn("dia_mes", dayofmonth("data_pedido")) \
             .withColumn("dia_semana", dayofweek("data_pedido"))
feats.select("pedido_id", "data_pedido", "mes", "trimestre", "dia_semana").show(5)

## 4. Conditional bucketing — ticket faixas

In [ ]:
from irispark.functions import when, lit

feats = feats.withColumn(
    "faixa_ticket",
    when(col("valor") < 100, lit("baixo"))
    .when(col("valor") < 400, lit("medio"))
    .otherwise(lit("alto")),
)
feats.groupBy("faixa_ticket").count().orderBy("faixa_ticket").show()

## 5. Window features — customer deltas & rolling sums

`lag` gives the previous order per customer; a 3-row frame gives a rolling sum.

In [ ]:
from irispark import Window
from irispark.functions import sum as s, lag

w = Window.partitionBy("cliente_id").orderBy("data_pedido")
w_roll = Window.partitionBy("cliente_id").orderBy("data_pedido").rowsBetween(-2, 0)

feats = feats.withColumn("valor_anterior", lag(col("valor"), 1).over(w)) \
             .withColumn("delta_valor", col("valor") - lag(col("valor"), 1).over(w)) \
             .withColumn("rolling_3_sum", s(col("valor")).over(w_roll))
feats.filter("cliente_id = 1").orderBy("data_pedido") \
     .select("data_pedido", "valor", "valor_anterior", "delta_valor", "rolling_3_sum").show()

## 6. ML transformer chain

Encode the category, scale the amount, assemble the feature vector — same fit/transform pattern as PySpark.

In [ ]:
from irispark.ml.feature import StringIndexer, OneHotEncoder, StandardScaler, VectorAssembler

idx = StringIndexer(inputCol="categoria", outputCol="cat_idx").fit(feats).transform(feats)
ohe = OneHotEncoder(inputCol="cat_idx", outputCol="cat_ohe").fit(idx).transform(idx)
scl = StandardScaler(inputCol="valor", outputCol="valor_std").fit(ohe).transform(ohe)
vec = VectorAssembler(inputCols=["valor_std", "cat_idx", "mes", "dia_semana"], outputCol="features")
features_df = vec.transform(scl)
features_df.select("pedido_id", "cat_idx", "valor_std", "features").show(5)

## 7. Materialize the training table

Persist the engineered set so training reads a table, not a pipeline.

In [ ]:
features_df.write.mode("overwrite").saveAsTable("ml_features_demo")
session.table("ml_features_demo").count()
print("materialized ml_features_demo")

## 8. Recipe transparency

`lineage` shows every transformation applied since the raw table.

In [ ]:
features_df.lineage(show=True)

## 9. Cleanup

In [ ]:
session.sql("DROP TABLE IF EXISTS ml_features_demo")
print("dropped ml_features_demo")

In [ ]:
if session is not None:
    session.close()
    print("Session closed.")